# 02 - Limpeza e Tratamento

Objetivo deste notebook:
- limpar e padronizar cada dataset bruto por fonte
- manter nomes de campos em pt-br
- usar `codigo_municipio + ano` como chave analitica principal
- salvar arquivos tratados para o notebook 03

Decisao de modelagem desta etapa:
- o projeto trabalha com capitais brasileiras
- a lista oficial de capitais sera usada como dimensao conformada inicial
- isso evita erro de municipios homonimos quando a origem possui apenas nome do municipio


## 1. Imports, caminhos e configuracao

In [18]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display

PASTA_DATASETS = Path('../datasets')
PASTA_IDH = PASTA_DATASETS / 'idh'
PASTA_CRIMES = PASTA_DATASETS / 'crimes'
PASTA_POPULACAO = PASTA_DATASETS / 'populacao'
PASTA_EDUCACAO = PASTA_DATASETS / 'educacao'
PASTA_TRATADOS = PASTA_DATASETS / 'tratados'

ANO_REFERENCIA_IDH = 2010
PASTA_TRATADOS.mkdir(parents=True, exist_ok=True)


## 2. Funcoes auxiliares

In [19]:
def listar_csvs(pasta: Path) -> list[Path]:
    if not pasta.exists():
        return []
    return sorted(pasta.glob('*.csv'))


def detectar_separador(caminho: Path) -> str:
    primeira_linha = caminho.read_text(encoding='utf-8-sig', errors='ignore').splitlines()[0]
    return ';' if primeira_linha.count(';') > primeira_linha.count(',') else ','


def ler_csv_padrao(caminho: Path) -> pd.DataFrame:
    df = pd.read_csv(
        caminho,
        sep=detectar_separador(caminho),
        encoding='utf-8-sig',
        decimal=',',
        low_memory=False,
    )
    return df.dropna(axis=1, how='all')


def normalizar_texto(texto: str) -> str:
    if pd.isna(texto):
        return None
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(caractere for caractere in texto if not unicodedata.combining(caractere))
    texto = re.sub(r'\s+', ' ', texto)
    return texto


def converter_colunas_numericas(df: pd.DataFrame, colunas: list[str]) -> pd.DataFrame:
    resultado = df.copy()
    for coluna in colunas:
        resultado[coluna] = (
            resultado[coluna]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .pipe(pd.to_numeric, errors='coerce')
        )
    return resultado


def validar_base(df: pd.DataFrame, chave: list[str], nome_dataset: str) -> None:
    print(f'Dataset tratado: {nome_dataset}')
    print(f'Linhas: {df.shape[0]}')
    print(f'Colunas: {df.shape[1]}')
    print(f'Duplicatas exatas: {df.duplicated().sum()}')
    print(f'Duplicatas na chave {chave}: {df.duplicated(subset=chave).sum()}')
    display(df.isna().sum().to_frame('qtd_nulos'))
    display(df.head())


## 3. Dimensao conformada inicial - capitais

In [20]:
capitais = [
    (1200401, 'Rio Branco', 'AC', 12),
    (2704302, 'Maceio', 'AL', 27),
    (1302603, 'Manaus', 'AM', 13),
    (1600303, 'Macapa', 'AP', 16),
    (2927408, 'Salvador', 'BA', 29),
    (2304400, 'Fortaleza', 'CE', 23),
    (5300108, 'Brasilia', 'DF', 53),
    (3205309, 'Vitoria', 'ES', 32),
    (5208707, 'Goiania', 'GO', 52),
    (2111300, 'Sao Luis', 'MA', 21),
    (3106200, 'Belo Horizonte', 'MG', 31),
    (5002704, 'Campo Grande', 'MS', 50),
    (5103403, 'Cuiaba', 'MT', 51),
    (1501402, 'Belem', 'PA', 15),
    (2507507, 'Joao Pessoa', 'PB', 25),
    (2611606, 'Recife', 'PE', 26),
    (2211001, 'Teresina', 'PI', 22),
    (4106902, 'Curitiba', 'PR', 41),
    (3304557, 'Rio de Janeiro', 'RJ', 33),
    (2408102, 'Natal', 'RN', 24),
    (1100205, 'Porto Velho', 'RO', 11),
    (1400100, 'Boa Vista', 'RR', 14),
    (4314902, 'Porto Alegre', 'RS', 43),
    (4205407, 'Florianopolis', 'SC', 42),
    (2800308, 'Aracaju', 'SE', 28),
    (3550308, 'Sao Paulo', 'SP', 35),
    (1721000, 'Palmas', 'TO', 17),
]

df_capitais = pd.DataFrame(
    capitais,
    columns=['codigo_municipio', 'nome_municipio_referencia', 'sigla_uf', 'codigo_uf_ibge']
)
df_capitais['nome_municipio_padronizado'] = df_capitais['nome_municipio_referencia'].apply(normalizar_texto)

display(df_capitais.head())
print('Total de capitais:', len(df_capitais))


,codigo_municipio,nome_municipio_referencia,sigla_uf,codigo_uf_ibge,nome_municipio_padronizado
0,1200401,Rio Branco,AC,12,rio branco
1,2704302,Maceio,AL,27,maceio
2,1302603,Manaus,AM,13,manaus
3,1600303,Macapa,AP,16,macapa
4,2927408,Salvador,BA,29,salvador


Total de capitais: 27


# Fonte 1 - IDH Municipal 2010

## 4.1 Leitura e padronizacao do IDH

In [21]:
ARQUIVO_IDH = listar_csvs(PASTA_IDH)[0]
df_idh_raw = ler_csv_padrao(ARQUIVO_IDH)

mapa_colunas_idh = {
    'Territorialidades': 'territorialidade',
    'Territorialidade': 'territorialidade',
    'IDHM 2010': 'idhm',
    'IDHM': 'idhm',
    'IDHM Renda 2010': 'idhm_renda',
    'IDHM Renda': 'idhm_renda',
    'IDHM Educação 2010': 'idhm_educacao',
    'IDHM Educação': 'idhm_educacao',
    'IDHM Longevidade 2010': 'idhm_longevidade',
    'IDHM Longevidade': 'idhm_longevidade',
}

df_idh = df_idh_raw.rename(columns=mapa_colunas_idh).copy()
df_idh['ano_referencia_idhm'] = ANO_REFERENCIA_IDH

df_idh['sigla_uf'] = df_idh['territorialidade'].astype(str).str.extract(r'\(([A-Z]{2})\)$')[0]
df_idh['nome_municipio'] = df_idh['territorialidade'].astype(str).str.replace(r'\s*\([A-Z]{2}\)$', '', regex=True)
df_idh = df_idh[df_idh['sigla_uf'].notna()].copy()
df_idh['nome_municipio_padronizado'] = df_idh['nome_municipio'].apply(normalizar_texto)

df_idh = converter_colunas_numericas(
    df_idh,
    ['idhm', 'idhm_renda', 'idhm_educacao', 'idhm_longevidade', 'ano_referencia_idhm']
)

df_idh_tratado = df_idh.merge(
    df_capitais[['codigo_municipio', 'codigo_uf_ibge', 'sigla_uf', 'nome_municipio_padronizado']],
    how='inner',
    on=['sigla_uf', 'nome_municipio_padronizado'],
    validate='one_to_one'
)

df_idh_tratado = df_idh_tratado[[
    'codigo_municipio',
    'codigo_uf_ibge',
    'sigla_uf',
    'nome_municipio',
    'nome_municipio_padronizado',
    'ano_referencia_idhm',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
    'territorialidade',
]].copy()

validar_base(df_idh_tratado, ['codigo_municipio'], 'IDH municipal 2010')


Dataset tratado: IDH municipal 2010
Linhas: 27
Colunas: 11
Duplicatas exatas: 0
Duplicatas na chave ['codigo_municipio']: 0


,qtd_nulos
codigo_municipio,0
codigo_uf_ibge,0
sigla_uf,0
nome_municipio,0
nome_municipio_padronizado,0
ano_referencia_idhm,0
idhm,0
idhm_renda,0
idhm_educacao,0
idhm_longevidade,0


,codigo_municipio,codigo_uf_ibge,sigla_uf,nome_municipio,nome_municipio_padronizado,ano_referencia_idhm,idhm,idhm_renda,idhm_educacao,idhm_longevidade,territorialidade
0,2800308,28,SE,Aracaju,aracaju,2010,0.770,0.784,0.708,0.823,Aracaju (SE)
1,1501402,15,PA,Belém,belem,2010,0.746,0.751,0.673,0.822,Belém (PA)
2,3106200,31,MG,Belo Horizonte,belo horizonte,2010,0.810,0.841,0.737,0.856,Belo Horizonte (MG)
3,1400100,14,RR,Boa Vista,boa vista,2010,0.752,0.737,0.708,0.816,Boa Vista (RR)
4,5300108,53,DF,Brasília,brasilia,2010,0.824,0.863,0.742,0.873,Brasília (DF)


In [22]:
df_idh_tratado.to_csv(PASTA_TRATADOS / 'idh_municipio_2010_tratado.csv', index=False)


# Fonte 2 - Crimes

## 5.1 Leitura e padronizacao de crimes

In [23]:
ARQUIVO_CRIMES = listar_csvs(PASTA_CRIMES)[0]
df_crimes_raw = ler_csv_padrao(ARQUIVO_CRIMES)

df_crimes = df_crimes_raw.rename(columns={
    'id_municipio': 'codigo_municipio',
    'id_municipio_nome': 'nome_municipio',
}).copy()

df_crimes['codigo_municipio'] = pd.to_numeric(df_crimes['codigo_municipio'], errors='coerce').astype('Int64')
df_crimes['ano'] = pd.to_numeric(df_crimes['ano'], errors='coerce').astype('Int64')
df_crimes['sigla_uf'] = df_crimes['sigla_uf'].astype(str).str.upper().str.strip()
df_crimes['nome_municipio_padronizado'] = df_crimes['nome_municipio'].apply(normalizar_texto)

colunas_quantidade_crimes = [coluna for coluna in df_crimes.columns if coluna.startswith('quantidade_')]
colunas_proporcao_crimes = [coluna for coluna in df_crimes.columns if coluna.startswith('proporcao_')]

df_crimes = converter_colunas_numericas(df_crimes, colunas_quantidade_crimes + colunas_proporcao_crimes)
df_crimes[colunas_quantidade_crimes] = df_crimes[colunas_quantidade_crimes].fillna(0)

df_crimes_tratado = df_crimes[df_crimes['codigo_municipio'].isin(df_capitais['codigo_municipio'])].copy()

df_crimes_tratado = df_crimes_tratado[[
    'codigo_municipio',
    'sigla_uf',
    'nome_municipio',
    'nome_municipio_padronizado',
    'ano',
    *colunas_quantidade_crimes,
    *colunas_proporcao_crimes,
]].copy()

validar_base(df_crimes_tratado, ['codigo_municipio', 'ano'], 'crimes municipio ano')


Dataset tratado: crimes municipio ano
Linhas: 162
Colunas: 31
Duplicatas exatas: 0
Duplicatas na chave ['codigo_municipio', 'ano']: 0


,qtd_nulos
codigo_municipio,0
sigla_uf,0
nome_municipio,0
nome_municipio_padronizado,0
ano,0
quantidade_mortes_intervencao_policial_civil_fora_de_servico,0
quantidade_feminicidio,0
quantidade_mortes_intervencao_policial_militar_fora_de_servico,0
quantidade_furto_veiculos,0
quantidade_mortes_intervencao_policial_civil_em_servico,0


,codigo_municipio,sigla_uf,nome_municipio,nome_municipio_padronizado,ano,quantidade_mortes_intervencao_policial_civil_fora_de_servico,quantidade_feminicidio,quantidade_mortes_intervencao_policial_militar_fora_de_servico,quantidade_furto_veiculos,quantidade_mortes_intervencao_policial_civil_em_servico,...,quantidade_roubo_furto_veiculos,quantidade_posse_ilegal_arma_de_fogo,quantidade_lesao_corporal_dolosa_violencia_domestica,quantidade_trafico_entorpecente,quantidade_roubo_veiculos,quantidade_lesao_corporal_morte,quantidade_morte_policiais_militares_confronto_em_servico,quantidade_posse_ilegal_porte_ilegal_arma_de_fogo,quantidade_homicidio_doloso,proporcao_mortes_intenvencao_policial_x_mortes_violentas_intencionais
0,2704302,AL,Maceió,maceio,2016,0.0,0.0,2.0,305.0,0.0,...,1538.0,0.0,0.0,0.0,1233.0,2.0,2.0,0.0,449,11.0
1,2304400,CE,Fortaleza,fortaleza,2016,0.0,0.0,7.0,2820.0,1.0,...,9235.0,0.0,0.0,0.0,6415.0,15.0,1.0,0.0,965,3.0
2,3205309,ES,Vitória,vitoria,2016,0.0,0.0,2.0,338.0,1.0,...,565.0,0.0,0.0,0.0,227.0,3.0,0.0,0.0,51,14.0
3,5208707,GO,Goiânia,goiania,2016,0.0,0.0,24.0,3733.0,0.0,...,11031.0,0.0,0.0,0.0,7298.0,14.0,0.0,0.0,452,16.0
4,2111300,MA,São Luís,sao luis,2016,0.0,0.0,0.0,487.0,0.0,...,2165.0,0.0,0.0,0.0,1678.0,12.0,0.0,0.0,498,4.0


In [24]:
df_crimes_tratado.to_csv(PASTA_TRATADOS / 'crimes_municipio_ano_tratado.csv', index=False)


# Fonte 3 - Populacao

## 6.1 Leitura, filtro de capitais e agregacao da populacao

In [25]:
ARQUIVO_POPULACAO = listar_csvs(PASTA_POPULACAO)[0]
df_populacao_raw = ler_csv_padrao(ARQUIVO_POPULACAO)

df_populacao = df_populacao_raw.rename(columns={
    'id_municipio': 'codigo_municipio',
    'id_municipio_nome': 'nome_municipio',
}).copy()

df_populacao['codigo_municipio'] = pd.to_numeric(df_populacao['codigo_municipio'], errors='coerce').astype('Int64')
df_populacao['ano'] = pd.to_numeric(df_populacao['ano'], errors='coerce').astype('Int64')
df_populacao['populacao'] = pd.to_numeric(df_populacao['populacao'], errors='coerce').fillna(0)
df_populacao['nome_municipio_padronizado'] = df_populacao['nome_municipio'].apply(normalizar_texto)

df_populacao_capitais = df_populacao[df_populacao['codigo_municipio'].isin(df_capitais['codigo_municipio'])].copy()

df_populacao_tratado = (
    df_populacao_capitais
    .groupby(['codigo_municipio', 'ano'], as_index=False)
    .agg(populacao_total=('populacao', 'sum'))
    .merge(df_capitais[['codigo_municipio', 'sigla_uf', 'codigo_uf_ibge', 'nome_municipio_referencia', 'nome_municipio_padronizado']], on='codigo_municipio', how='left')
    .rename(columns={'nome_municipio_referencia': 'nome_municipio'})
)

df_populacao_tratado = df_populacao_tratado.sort_values(['codigo_municipio', 'ano']).copy()
df_populacao_tratado['populacao_crescimento_pct'] = (
    df_populacao_tratado
    .groupby('codigo_municipio')['populacao_total']
    .pct_change()
    .mul(100)
)

df_populacao_tratado = df_populacao_tratado[[
    'codigo_municipio',
    'codigo_uf_ibge',
    'sigla_uf',
    'nome_municipio',
    'nome_municipio_padronizado',
    'ano',
    'populacao_total',
    'populacao_crescimento_pct',
]].copy()

validar_base(df_populacao_tratado, ['codigo_municipio', 'ano'], 'populacao municipio ano')


Dataset tratado: populacao municipio ano
Linhas: 162
Colunas: 8
Duplicatas exatas: 0
Duplicatas na chave ['codigo_municipio', 'ano']: 0


,qtd_nulos
codigo_municipio,0
codigo_uf_ibge,0
sigla_uf,0
nome_municipio,0
nome_municipio_padronizado,0
ano,0
populacao_total,0
populacao_crescimento_pct,27


,codigo_municipio,codigo_uf_ibge,sigla_uf,nome_municipio,nome_municipio_padronizado,ano,populacao_total,populacao_crescimento_pct
0,1100205,11,RO,Porto Velho,porto velho,2016,499293,NaN
1,1100205,11,RO,Porto Velho,porto velho,2017,509323,2.008841
2,1100205,11,RO,Porto Velho,porto velho,2018,519531,2.004229
3,1100205,11,RO,Porto Velho,porto velho,2019,529544,1.927315
4,1100205,11,RO,Porto Velho,porto velho,2020,539354,1.852537


In [26]:
df_populacao_tratado.to_csv(PASTA_TRATADOS / 'populacao_municipio_ano_tratado.csv', index=False)


# Fonte 4 - Educacao / IDEB

## 7.1 Leitura e padronizacao da educacao

In [27]:
ARQUIVO_EDUCACAO = listar_csvs(PASTA_EDUCACAO)[0]
df_educacao_raw = ler_csv_padrao(ARQUIVO_EDUCACAO)

df_educacao = df_educacao_raw.copy()
df_educacao = converter_colunas_numericas(
    df_educacao,
    ['ibge_id', 'dependencia_id', 'ano', 'ideb', 'fluxo', 'aprendizado', 'nota_mt', 'nota_lp']
)

df_educacao = df_educacao.rename(columns={'ibge_id': 'codigo_uf_ibge'}).copy()
df_educacao['codigo_uf_ibge'] = df_educacao['codigo_uf_ibge'].astype('Int64')
df_educacao['dependencia_id'] = df_educacao['dependencia_id'].astype('Int64')
df_educacao['ano'] = df_educacao['ano'].astype('Int64')
df_educacao['ciclo_id'] = df_educacao['ciclo_id'].astype(str).str.upper().str.strip()

df_ufs = df_capitais[['codigo_uf_ibge', 'sigla_uf']].drop_duplicates()
df_educacao_tratado = df_educacao.merge(df_ufs, how='left', on='codigo_uf_ibge', validate='many_to_one')

df_educacao_tratado = df_educacao_tratado[[
    'codigo_uf_ibge',
    'sigla_uf',
    'dependencia_id',
    'ciclo_id',
    'ano',
    'ideb',
    'fluxo',
    'aprendizado',
    'nota_mt',
    'nota_lp',
]].copy()

validar_base(df_educacao_tratado, ['codigo_uf_ibge', 'dependencia_id', 'ciclo_id', 'ano'], 'educacao uf ano')


Dataset tratado: educacao uf ano
Linhas: 243
Colunas: 10
Duplicatas exatas: 0
Duplicatas na chave ['codigo_uf_ibge', 'dependencia_id', 'ciclo_id', 'ano']: 0


,qtd_nulos
codigo_uf_ibge,0
sigla_uf,0
dependencia_id,0
ciclo_id,0
ano,0
ideb,0
fluxo,0
aprendizado,0
nota_mt,0
nota_lp,0


,codigo_uf_ibge,sigla_uf,dependencia_id,ciclo_id,ano,ideb,fluxo,aprendizado,nota_mt,nota_lp
0,11,RO,0,EM,2017,4.0,0.8735,4.5271,272.03,268.33
1,11,RO,2,EM,2017,3.8,0.8498,4.4200,268.05,264.91
2,11,RO,4,EM,2017,5.5,0.9496,5.7899,319.00,308.62
3,12,AC,0,EM,2017,3.8,0.8770,4.3461,263.28,264.45
4,12,AC,2,EM,2017,3.6,0.8476,4.2562,260.01,261.51


## 7.2 Criar visao educacional total por UF e ano

In [28]:
df_educacao_uf_ano_total = df_educacao_tratado[
    (df_educacao_tratado['dependencia_id'] == 0) &
    (df_educacao_tratado['ciclo_id'] == 'EM')
].copy()

df_educacao_uf_ano_total = df_educacao_uf_ano_total[[
    'codigo_uf_ibge',
    'sigla_uf',
    'ano',
    'ideb',
    'fluxo',
    'aprendizado',
    'nota_mt',
    'nota_lp',
]].copy()

validar_base(df_educacao_uf_ano_total, ['codigo_uf_ibge', 'ano'], 'educacao total EM uf ano')


Dataset tratado: educacao total EM uf ano
Linhas: 81
Colunas: 8
Duplicatas exatas: 0
Duplicatas na chave ['codigo_uf_ibge', 'ano']: 0


,qtd_nulos
codigo_uf_ibge,0
sigla_uf,0
ano,0
ideb,0
fluxo,0
aprendizado,0
nota_mt,0
nota_lp,0


,codigo_uf_ibge,sigla_uf,ano,ideb,fluxo,aprendizado,nota_mt,nota_lp
0,11,RO,2017,4.0,0.8735,4.5271,272.03,268.33
3,12,AC,2017,3.8,0.8770,4.3461,263.28,264.45
6,13,AM,2017,3.5,0.8745,4.0231,250.93,254.46
9,14,RR,2017,3.5,0.8457,4.1174,256.73,255.32
12,15,PA,2017,3.1,0.8035,3.8280,246.29,245.78


In [29]:
df_educacao_tratado.to_csv(PASTA_TRATADOS / 'educacao_ideb_uf_ano_tratado.csv', index=False)
df_educacao_uf_ano_total.to_csv(PASTA_TRATADOS / 'educacao_ideb_uf_ano_total_em_tratado.csv', index=False)


# Saidas deste notebook

In [30]:
arquivos_gerados = sorted(PASTA_TRATADOS.glob('*.csv'))
for arquivo in arquivos_gerados:
    print(arquivo)


../datasets/tratados/crimes_municipio_ano_tratado.csv
../datasets/tratados/educacao_ideb_uf_ano_total_em_tratado.csv
../datasets/tratados/educacao_ideb_uf_ano_tratado.csv
../datasets/tratados/idh_municipio_2010_tratado.csv
../datasets/tratados/populacao_municipio_ano_tratado.csv


Dataframes tratados gerados:
- `df_idh_tratado`: IDHM das capitais, referencia 2010
- `df_crimes_tratado`: indicadores criminais por capital e ano
- `df_populacao_tratado`: populacao total por capital e ano
- `df_educacao_tratado`: IDEB por UF, dependencia, ciclo e ano
- `df_educacao_uf_ano_total`: IDEB total do Ensino Medio por UF e ano

Esses arquivos entram no notebook 03 para integracao e feature engineering.
